In [0]:
from pyspark.sql.functions import col, collect_set, concat_ws, current_timestamp

print("Starting Gold Layer Pipeline...")

# GOLD DIMENSION TABLE: gold.dim_patient
print("Creating gold.dim_patient...")

df_silver_patient = spark.read.table("silver.patient").filter("is_current = true")

df_gold_dim_patient = df_silver_patient.select(
    col("patient_id"),
    col("gender"),
    col("birthDate"),
    col("effective_date").alias("record_effective_date")
)

df_gold_dim_patient.write.format("delta").mode("overwrite").saveAsTable("gold.dim_patient")
print("Created table: gold.dim_patient")


# GOLD FACT TABLE: gold.fact_patient_encounters
print("Creating gold.fact_patient_encounters...")

# Read Active Silver Tables
df_p = spark.read.table("silver.patient").filter("is_current = true")
df_e = spark.read.table("silver.encounter").filter("is_current = true")
df_c = spark.read.table("silver.condition").filter("is_current = true")
df_o = spark.read.table("silver.observation").filter("is_current = true")

# Pre-aggregate child tables to 1 row per patient_id to guarantee fast, clean joins
e_agg = df_e.groupBy("patient_id").agg(
    concat_ws(", ", collect_set("encounter_id")).alias("encounter_ids"),
    concat_ws(", ", collect_set("status")).alias("encounter_statuses")
)

c_agg = df_c.groupBy("patient_id").agg(
    concat_ws(", ", collect_set("condition_id")).alias("condition_ids"),
    concat_ws(", ", collect_set("clinical_status")).alias("clinical_statuses")
)

o_agg = df_o.groupBy("patient_id").agg(
    concat_ws(", ", collect_set("observation_id")).alias("observation_ids"),
    concat_ws(", ", collect_set("status")).alias("observation_statuses")
)

# Broadcast / Equi-Join across aggregated DataFrames
df_gold_fact = df_p.alias("p") \
    .join(e_agg.alias("e"), col("p.patient_id") == col("e.patient_id"), "left") \
    .join(c_agg.alias("c"), col("p.patient_id") == col("c.patient_id"), "left") \
    .join(o_agg.alias("o"), col("p.patient_id") == col("o.patient_id"), "left") \
    .select(
        col("p.patient_id"),
        col("p.gender"),
        col("p.birthDate"),
        col("e.encounter_ids"),
        col("e.encounter_statuses"),
        col("c.condition_ids"),
        col("c.clinical_statuses"),
        col("o.observation_ids"),
        col("o.observation_statuses")
    )

df_gold_fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_patient_encounters")
print("Created table: gold.fact_patient_encounters")

print("\nGold Layer Processing Complete!")

Starting Gold Layer Pipeline...
Creating gold.dim_patient...
Created table: gold.dim_patient
Creating gold.fact_patient_encounters...
Created table: gold.fact_patient_encounters

Gold Layer Processing Complete!
